In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

**Notebook diagnostic:** Run the next code cell to show which Python executable this notebook kernel is using and whether numpy is importable. If numpy is missing, run the third cell to install it into this kernel.

In [ ]:
def generate_dataset(n, seed=42):
    np.random.seed(seed)

    # Dog physiology
    Weight = np.random.normal(20, 7, n).clip(5, 45)
    Age = np.random.normal(5, 2, n).clip(0.5, 14)

    RBC = np.random.normal(6.5, 0.8, n).clip(4.5, 8.5)
    HB = np.random.normal(14, 1.8, n).clip(8, 20)
    Creatinine = np.random.normal(1.0, 0.25, n).clip(0.4, 2.5)
    Glucose = np.random.normal(100, 18, n).clip(60, 180)

    Dose = np.random.uniform(0.2, 2.5, n)

    # Therapeutic window baseline
    lower_safe = 0.5
    upper_safe = 2.0

    # Dose-based safety score (smooth, not binary)
    dose_score = np.where(
        Dose < lower_safe,
        -1.0,
        np.where(Dose > upper_safe, -1.0, 1.0)
    )

    # Hematology modifier (individual tolerance)
    phys_score = (
        -0.6 * (Creatinine / 2.5) +
        0.5 * (HB / 20) -
        0.3 * (Glucose / 180)
    )

    total_score = dose_score + phys_score

    # Add biological noise
    total_score += np.random.normal(0, 0.4, n)

    # Convert to probability
    prob_safe = 1 / (1 + np.exp(-total_score))

    Outcome = (prob_safe > 0.5).astype(int)

    return pd.DataFrame({
        "Weight": Weight,
        "Age": Age,
        "RBC": RBC,
        "HB": HB,
        "Creatinine": Creatinine,
        "Glucose": Glucose,
        "Dose_mg_per_kg": Dose,
        "Outcome": Outcome
    })

In [ ]:
# Load full risky to safe database
df_1250 = pd.read_csv('smart_vet_dose_1250.csv')
df_1250.head()

In [ ]:
# Model Evaluation Function for Random Forest Only
def evaluate_random_forest(df):
    X = df.drop("Outcome", axis=1)
    y = df["Outcome"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    # Random Forest
    rf = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    results = {
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Precision": precision_score(y_test, rf_pred),
        "Recall": recall_score(y_test, rf_pred),
        "F1": f1_score(y_test, rf_pred)
    }
    return results, rf, X_test, y_test

model_results, rf_model, X_test, y_test = evaluate_random_forest(df_1250)
print("Amount of Precision and Model Metrics:")
for metric, value in model_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Calculate recommended safe dosage level
safe_data = df_1250[df_1250['Outcome'] == 1]
safe_doses = safe_data['Dose_mg_per_kg']

mean_dose = safe_doses.mean()
std_dose = safe_doses.std()
n = len(safe_doses)

# 95% Confidence Interval for the mean
z = 1.96
margin_of_error = z * (std_dose / np.sqrt(n))

lower_bound = mean_dose - margin_of_error
upper_bound = mean_dose + margin_of_error

print("--- Pharmacokinetic Validation ---")
print(f"Recommended safe dosage interval: {lower_bound:.2f} to {upper_bound:.2f} mg/kg")

In [ ]:
# Confusion Matrix Visualization for Random Forest
y_pred = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Random Forest")
plt.show()
